# Solow Model: Impulse Response Functions (IRFs)

This notebook reproduces the Chapter 3 IRF exercise for a temporary technology shock in the Solow model:

- Nonlinear law of motion: $k_{t+1} = s A_t k_t^{\alpha} \ell^{1-\alpha} + (1-\delta)k_t$
- Shock path: $A_0=(1+\varepsilon)\bar A$, and for $t\ge 1$, $A_t=(1+\rho^t\varepsilon)\bar A$
- Comparison with log-linear approximation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Baseline parameters (matching the lecture example)
alpha = 1/3
s = 0.2
delta = 0.046
ell = 1.0
A_bar = 1.0
eps = 0.01      # 1% initial shock
rho = 0.9       # shock persistence
T = 80          # horizon

# Steady state before the shock (nonlinear model)
k_bar = (s * A_bar * ell**(1-alpha) / delta) ** (1 / (1-alpha))
y_bar = A_bar * (k_bar**alpha) * (ell**(1-alpha))

print(f'k_bar = {k_bar:.6f}')
print(f'y_bar = {y_bar:.6f}')

In [ ]:
# Build technology path A_t
A = np.empty(T + 1)
A[0] = (1 + eps) * A_bar
for t in range(1, T + 1):
    A[t] = (1 + (rho**t) * eps) * A_bar

# Simulate nonlinear dynamics
k = np.empty(T + 1)
y = np.empty(T + 1)
k[0] = k_bar
y[0] = A[0] * (k[0]**alpha) * (ell**(1-alpha))

for t in range(T):
    k[t+1] = s * A[t] * (k[t]**alpha) * (ell**(1-alpha)) + (1-delta) * k[t]
    y[t+1] = A[t+1] * (k[t+1]**alpha) * (ell**(1-alpha))

# Percent deviations from steady state
k_pct = 100 * (k / k_bar - 1)
y_pct = 100 * (y / y_bar - 1)
A_pct = 100 * (A / A_bar - 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(A_pct, label='A shock (%)', lw=2)
ax[0].plot(k_pct, label='k response (%)', lw=2)
ax[0].set_title('Technology Shock and Capital IRF')
ax[0].set_xlabel('t')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(y_pct, label='y response (%)', lw=2, color='tab:green')
ax[1].set_title('Output IRF (Nonlinear Simulation)')
ax[1].set_xlabel('t')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Log-linear approximation around steady state
# khat_{t+1} = (1 - lambda) khat_t + delta * Ahat_t, lambda = delta(1-alpha)
lam = delta * (1 - alpha)

Ahat = np.log(A / A_bar)
khat_lin = np.empty(T + 1)
khat_lin[0] = 0.0
for t in range(T):
    khat_lin[t+1] = (1 - lam) * khat_lin[t] + delta * Ahat[t]

yhat_lin = Ahat + alpha * khat_lin

# Convert linearized log deviations to percent
k_lin_pct = 100 * khat_lin
y_lin_pct = 100 * yhat_lin

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(k_pct, label='k nonlinear (%)', lw=2)
ax[0].plot(k_lin_pct, '--', label='k log-linear (%)', lw=2)
ax[0].set_title('Capital IRF: Nonlinear vs Log-linear')
ax[0].set_xlabel('t')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(y_pct, label='y nonlinear (%)', lw=2)
ax[1].plot(y_lin_pct, '--', label='y log-linear (%)', lw=2)
ax[1].set_title('Output IRF: Nonlinear vs Log-linear')
ax[1].set_xlabel('t')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

max_abs_err_k = np.max(np.abs(k - k_bar * np.exp(khat_lin)))
print(f'Max abs level error in k (nonlinear vs log-linear): {max_abs_err_k:.6e}')

In [ ]:
# Optional: quick experiment helper
def simulate_irf(alpha=1/3, s=0.2, delta=0.046, ell=1.0, eps=0.01, rho=0.9, T=80):
    A_bar = 1.0
    k_bar = (s * A_bar * ell**(1-alpha) / delta) ** (1 / (1-alpha))
    y_bar = A_bar * k_bar**alpha * ell**(1-alpha)

    A = np.empty(T+1)
    A[0] = (1 + eps) * A_bar
    for t in range(1, T+1):
        A[t] = (1 + (rho**t) * eps) * A_bar

    k = np.empty(T+1)
    y = np.empty(T+1)
    k[0] = k_bar
    y[0] = A[0] * k[0]**alpha * ell**(1-alpha)

    for t in range(T):
        k[t+1] = s * A[t] * k[t]**alpha * ell**(1-alpha) + (1-delta) * k[t]
        y[t+1] = A[t+1] * k[t+1]**alpha * ell**(1-alpha)

    return {
        'A_pct': 100 * (A / A_bar - 1),
        'k_pct': 100 * (k / k_bar - 1),
        'y_pct': 100 * (y / y_bar - 1),
        'k_bar': k_bar,
        'y_bar': y_bar
    }

res = simulate_irf(eps=0.02, rho=0.95, T=100)
plt.figure(figsize=(7,4))
plt.plot(res['A_pct'], label='A shock (%)')
plt.plot(res['k_pct'], label='k response (%)')
plt.plot(res['y_pct'], label='y response (%)')
plt.title('IRF Experiment (eps=2%, rho=0.95)')
plt.xlabel('t')
plt.legend()
plt.grid(alpha=0.3)
plt.show()